In [ ]:
import dlt
from pyspark.sql import functions as F

SOURCE_PATH = "/Volumes/workspace/default/my_volume/WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [ ]:
# Bronze layer: persist the raw data exactly as read from the volume
@dlt.table(name="telco_customer_churn_bronze", comment="Raw Telco churn data, as ingested")
def telco_customer_churn_bronze():
    return spark.read.csv(SOURCE_PATH, header=True, inferSchema=True, sep=",")

In [ ]:
# Silver layer: cleaned data — remove the Partner column and drop rows with any null value
@dlt.table(name="telco_customer_churn_silver", comment="Cleaned churn data used for training and scoring")
def telco_customer_churn_silver():
    return dlt.read("telco_customer_churn_bronze").drop("Partner").na.drop()

In [ ]:
# Gold layer: compact churn summary per segment
@dlt.table(name="telco_customer_churn_gold", comment="Churn rate and average charges by customer segment")
def telco_customer_churn_gold():
    silver_df = dlt.read("telco_customer_churn_silver")

    # Bucket raw tenure months into readable lifecycle segments
    silver_df = silver_df.withColumn(
        "tenure_bucket",
        F.when(F.col("tenure") <= 12, "0-12")
         .when(F.col("tenure") <= 24, "13-24")
         .when(F.col("tenure") <= 48, "25-48")
         .otherwise("49-72"),
    )

    agg_exprs = [
        F.count("*").alias("customers"),
        F.round(F.avg(F.when(F.col("Churn") == "Yes", 1.0).otherwise(0.0)) * 100, 2).alias("churn_rate_pct"),
        F.round(F.avg("MonthlyCharges"), 2).alias("avg_monthly_charges"),
    ]

    # Core business segments: contract type, internet service, tenure, and age — the biggest churn drivers
    segments = {
        "contract": "Contract",
        "internet_service": "InternetService",
        "tenure": "tenure_bucket",
        "age": "SeniorCitizen",
    }

    gold_df = None
    for segment_name, col in segments.items():
        seg_df = (silver_df.groupBy(col).agg(*agg_exprs)
                  .withColumn("segment_type", F.lit(segment_name))
                  .withColumnRenamed(col, "segment_value")
                  .withColumn("segment_value", F.col("segment_value").cast("string")))
        gold_df = seg_df if gold_df is None else gold_df.unionByName(seg_df)

    return gold_df

# Output explanation:
# - segment_type   : which dimension the row aggregates (contract, internet_service, tenure, age)
# - segment_value  : the specific value within that dimension (e.g. 'Month-to-month', '0-12', '1')
# - customers      : number of customers in that segment
# - churn_rate_pct : % of those customers who churned (Churn == 'Yes')
# - avg_monthly_charges : average MonthlyCharges for the segment
# Each dimension's rows partition the full silver population, so segment rows
# within one segment_type sum back to the total customer count.